# 02 - Embeddings Profundos com CelebA Align e StyleGAN2

Neste notebook vamos gerar embeddings com `ResNet50`, visualizar a distribuicao em 2D e salvar o modelo padrao considerando **as imagens originais e as imagens sinteticas**.


In [ ]:
import pickle
from math import ceil
import random
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ResNet50_Weights, resnet50

try:
    import umap
except ImportError:
    umap = None

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATASET_ROOT = Path('/home/arthur/Documentos/Github/Projeto-5-Redes-Neurais/data/raw/img_align_celeba')
SYNTHETIC_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'stylegan2_synthetic_256'
OUTPUT_PATH = PROJECT_ROOT / 'artifacts' / 'embeddings' / 'celeba_align_plus_synthetic_deep_resnet50.parquet'
MODEL_OUTPUT_PATH = PROJECT_ROOT / 'artifacts' / 'models' / 'deep_resnet50_model.pkl'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
SAMPLE_FRACTION = 0.15
SAMPLE_RANDOM_SEED = 42
MAX_SAMPLES_PER_SOURCE = None
IMAGE_SIZE = 256

for folder in [OUTPUT_PATH.parent, MODEL_OUTPUT_PATH.parent, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('Originais em:', DATASET_ROOT)
print('Sinteticas em:', SYNTHETIC_ROOT)
print('Fracao da amostra:', SAMPLE_FRACTION)
print('Sample random seed:', SAMPLE_RANDOM_SEED)


In [ ]:
def build_combined_manifest(original_root, synthetic_root=None, sample_fraction=0.20, sample_random_seed=42, max_samples_per_source=None):
    sources = [
        ('original', Path(original_root), '*.jpg'),
        ('synthetic', Path(synthetic_root), '*.png'),
    ]

    frames = []
    for source_name, root, pattern in sources:
        if root is None or not root.exists():
            continue
        images = sorted(root.glob(pattern))
        if images:
            sample_size = min(len(images), ceil(len(images) * sample_fraction))
            rng = random.Random(sample_random_seed)
            images = sorted(rng.sample(images, sample_size))
        if max_samples_per_source:
            images = images[:max_samples_per_source]
        if images:
            frames.append(pd.DataFrame({
                'image_name': [path.name for path in images],
                'image_path': [str(path) for path in images],
                'source': source_name,
            }))

    if not frames:
        raise FileNotFoundError('Nenhuma imagem original ou sintetica foi encontrada.')

    return pd.concat(frames, ignore_index=True)

class AlignCelebADataset(Dataset):
    def __init__(self, original_root, synthetic_root=None, image_size=256, sample_fraction=0.20, sample_random_seed=42, max_samples_per_source=None):
        self.samples = build_combined_manifest(
            original_root=original_root,
            synthetic_root=synthetic_root,
            sample_fraction=sample_fraction,
            sample_random_seed=sample_random_seed,
            max_samples_per_source=max_samples_per_source,
        )
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        row = self.samples.iloc[index]
        image = Image.open(row['image_path']).convert('RGB')
        return {
            'image': self.transform(image),
            'image_name': row['image_name'],
            'image_path': row['image_path'],
            'source': row['source'],
        }

def build_resnet50_embedding_model():
    model = resnet50(weights=ResNet50_Weights.DEFAULT)
    model.fc = nn.Identity()
    model.eval()
    return model

def extract_deep_embeddings(original_root, synthetic_root, output_path, batch_size=32, num_workers=0, sample_fraction=0.20, sample_random_seed=42, max_samples_per_source=None, image_size=256, device=None):
    dataset = AlignCelebADataset(
        original_root=original_root,
        synthetic_root=synthetic_root,
        image_size=image_size,
        sample_fraction=sample_fraction,
        sample_random_seed=sample_random_seed,
        max_samples_per_source=max_samples_per_source,
    )
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    resolved_device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    model = build_resnet50_embedding_model().to(resolved_device)

    rows = []
    with torch.no_grad():
        for batch in dataloader:
            images = batch['image'].to(resolved_device)
            embeddings = model(images).cpu().numpy()
            image_names = batch['image_name']
            image_paths = batch['image_path']
            sources = batch['source']
            for embedding, image_name, image_path, source in zip(embeddings, image_names, image_paths, sources):
                row = {'image_name': image_name, 'image_path': image_path, 'source': source}
                for index, value in enumerate(embedding):
                    row[f'f_{index:04d}'] = float(value)
                rows.append(row)

    frame = pd.DataFrame(rows)
    frame.to_parquet(output_path, index=False)
    return frame

def feature_matrix(frame):
    return frame.drop(columns=['image_name', 'image_path', 'source'])

def reduce_embeddings(frame, method='pca', random_state=42):
    features = feature_matrix(frame)
    if method == 'pca':
        reducer = PCA(n_components=2, random_state=random_state)
    elif method == 'tsne':
        reducer = TSNE(n_components=2, random_state=random_state, init='pca')
    elif method == 'umap':
        if umap is None:
            raise ImportError("Instale 'umap-learn' para usar UMAP.")
        reducer = umap.UMAP(n_components=2, random_state=random_state)
    else:
        raise ValueError(f'Metodo desconhecido: {method}')
    reduced = reducer.fit_transform(features)
    result = frame[['image_name', 'image_path', 'source']].copy()
    result['x'] = reduced[:, 0]
    result['y'] = reduced[:, 1]
    result['method'] = method
    return result

def save_scatter_plot(frame, output_path, title, hue='source'):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=frame, x='x', y='y', hue=hue, alpha=0.75, s=40)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()

def save_pickle_model(model, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'wb') as file:
        pickle.dump(model, file)
    return output_path

def save_default_resnet50_model(output_path):
    model = build_resnet50_embedding_model().cpu()
    return save_pickle_model(model, output_path)


## Extracao dos embeddings profundos

In [ ]:
deep_df = extract_deep_embeddings(
    original_root=DATASET_ROOT,
    synthetic_root=SYNTHETIC_ROOT,
    output_path=OUTPUT_PATH,
    sample_fraction=SAMPLE_FRACTION,
    sample_random_seed=SAMPLE_RANDOM_SEED,
    max_samples_per_source=MAX_SAMPLES_PER_SOURCE,
    image_size=IMAGE_SIZE,
)
deep_df.head()


## Reducao para 2D

In [ ]:
deep_df = pd.read_parquet(OUTPUT_PATH)
deep_2d = reduce_embeddings(deep_df, method='pca')
deep_2d.head()


In [ ]:
figure_path = FIGURES_DIR / 'celeba_align_deep_pca.png'
save_scatter_plot(deep_2d, figure_path, 'CelebA Align - Deep Embeddings (PCA)')
figure_path

In [ ]:
model_path = save_default_resnet50_model(MODEL_OUTPUT_PATH)
print('Modelo ResNet50 padrao salvo em:', model_path)

In [ ]:
deep_df

In [ ]:
deep_features = feature_matrix(deep_df)
deep_labels = (deep_df['source'] == 'synthetic').astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    deep_features,
    deep_labels,
    test_size=0.20,
    random_state=SAMPLE_RANDOM_SEED,
    stratify=deep_labels,
)

deep_tree_classifier = DecisionTreeClassifier(
    random_state=SAMPLE_RANDOM_SEED,
    max_depth=12,
    min_samples_leaf=5,
)
deep_tree_classifier.fit(X_train, y_train)
deep_predictions = deep_tree_classifier.predict(X_test)
deep_accuracy = accuracy_score(y_test, deep_predictions)

deep_metrics = {
    'metric': 'source_classification',
    'classifier': 'decision_tree',
    'accuracy': float(deep_accuracy),
    'error_rate': float(1.0 - deep_accuracy),
    'train_samples': int(len(X_train)),
    'test_samples': int(len(X_test)),
}

print('Classificador de arvore para embeddings profundos: DecisionTreeClassifier')
print('Accuracy (original vs synthetic):', round(deep_metrics['accuracy'], 4))
print('Error rate:', round(deep_metrics['error_rate'], 4))
pd.DataFrame([deep_metrics])